# Thêm Thư Viện

In [6]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [7]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

# Đọc data

## Đọc data từ SQL Server

In [8]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY So_the
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

In [9]:
query_Bandoc = """
SELECT dbo.DecodeUTF8String(So_the) AS So_the,
       dbo.DecodeUTF8String(Ho_ten) AS Ho_ten,
       Ngay_sinh,
       Dan_toc_ID,
       Trinh_do_ID,
       dbo.DecodeUTF8String(So_dien_thoai) AS So_dien_thoai,
       dbo.DecodeUTF8String(Nghe_nghiep) AS Nghe_nghiep,
       dbo.DecodeUTF8String(Co_quan) AS Co_quan,
       dbo.DecodeUTF8String(chuc_vu) AS chuc_vu,
       dbo.DecodeUTF8String(Dia_chi) AS Dia_chi,
       dbo.DecodeUTF8String(Dia_chi_thuong_tru) AS Dia_chi_thuong_tru,
       dbo.DecodeUTF8String(Khoa_hoc) AS Khoa_hoc,
       Lop,
       Anh,
       Ngay_cap,
       Ngay_het_han,
       Email,
       Nhom_ID,
       Nhom_nghanh_nghe_ID,
       Gioi_tinh,
       Status,
       dbo.DecodeUTF8String(Ghi_chu) AS Ghi_chu,
       Mat_khau
FROM Ban_doc
"""
df_bandoc = fetch_data_in_batches(query_Bandoc, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_bandoc) # Hiển thị kết quả

C:\Users\admin\AppData\Local\Temp\ipykernel_20680\705386798.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


Lỗi xảy ra khi xử lý batch từ 58900: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 59200: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 59300: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 59400: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 59800: ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]XML parsing: line 1, character 78, illegal xml character (9420) (SQLFetch)')
Lỗi xảy ra khi xử lý batch từ 60000: ('42000', '[42000] [Microsof

C:\Users\admin\AppData\Local\Temp\ipykernel_20680\705386798.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()


### Tạo dataframe backup 

In [10]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_bandoc.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_bandoc_backup = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_bandoc:", len(df_bandoc)) # Kiểm tra số lượng dòng
print("Số dòng trong df_bandoc_backup:", len(df_bandoc_backup)) # Kiểm tra số lượng dòng

Số dòng trong df_bandoc: 71083
Số dòng trong df_bandoc_backup: 71083


### [Nếu cần] lấy lại data từ backup

In [67]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_bandoc_backup.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_bandoc = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_bandoc_backup:", len(df_bandoc_backup)) # Kiểm tra số lượng dòng
print("Số dòng trong df_bandoc:", len(df_bandoc)) # Kiểm tra số lượng dòng

Số dòng trong df_bandoc_backup: 71083
Số dòng trong df_bandoc: 71083


# Xử lý data

## Thêm 1 dòng giả định none

In [68]:
# Tạo DataFrame `new_row` chứa dòng dữ liệu giả định
new_row = pd.DataFrame({
    'So_the': ['0'],
    'Ho_ten': ['(Không xác định)'],
    'Ngay_sinh': ['1024-01-01 00:00:00'],
    'Khoa_hoc': [0],
    'Dan_toc_ID': [56],
    'Trinh_do_ID': [0],
    'Lop': [0],
    'Ngay_cap': ['1024-01-01 00:00:00'],
    'Ngay_het_han': ['1024-01-01 00:00:00'],
    'Nhom_ID': [0],
    'Nhom_nghanh_nghe_ID': [0],
})

# Thêm dòng dữ liệu giả định vào `df` bằng `pd.concat`
df_bandoc = pd.concat([df_bandoc, new_row], ignore_index=True) # Thêm vào dataframe
df_bandoc['So_the'] = df_bandoc['So_the'].astype(str)
df_bandoc = df_bandoc.sort_values(by="So_the", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_bandoc)

           So_the            Ho_ten            Ngay_sinh  Dan_toc_ID  \
0               0  (Không xác định)  1024-01-01 00:00:00          56   
1       0/8107029       Vũ Minh Đức  1989-05-02 00:00:00           1   
2        00000001    LƯU VĨNH QUANG  1976-01-01 00:00:00           1   
3        00000001    LƯU VĨNH QUANG  1976-01-01 00:00:00           1   
4        00000002     PHẠM HỮU TIẾN  1992-02-15 00:00:00           1   
...           ...               ...                  ...         ...   
71079   ÊU0401079       LÊ VĂN HIẾU                  NaT           1   
71080    Ô0105040        NG. TẤN ĐỘ                  NaT           1   
71081   ên2101158      ĐÀO MINH ĐỨC                  NaT           1   
71082   ƠI4401092      PHẠM VĂN LỢI  1980-05-31 00:00:00           1   
71083  ƠNG1113054  LÊ THỊ MỸ PHƯỢNG  1982-08-06 00:00:00           1   

       Trinh_do_ID So_dien_thoai Nghe_nghiep  \
0                0           NaN         NaN   
1                4                     

## Xử lý data rỗng hoặc " "

In [69]:
df_bandoc = df_bandoc.replace(np.nan, None)
df_bandoc = df_bandoc.replace('', None)
print(df_bandoc[['Nghe_nghiep', 'Co_quan', 'chuc_vu']])

      Nghe_nghiep                                 Co_quan chuc_vu
0            None                                    None    None
1            None                           Trường ĐHSPKT    None
2            None                           Trường ĐHSPKT    None
3            None  Trường TCN AN ĐỨC- TỔNG CONG TY LIKSIN    None
4            None                           Trường ĐHSPKT    None
...           ...                                     ...     ...
71079        None                                  ĐHSPKT    None
71080        None                                  ĐHSPKT    None
71081        None                                  ĐHSPKT    None
71082        None                                  ĐHSPKT    None
71083        None                                  ĐHSPKT    None

[71084 rows x 3 columns]


## Xử lý kiểu date

In [70]:
query_date = "SELECT Date_key FROM olap.DIM_date"
df_date = pd.read_sql(query_date, conn_dwh_library)

date_ids = set(df_date['Date_key'])

# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_bandoc['Ngay_sinh'] = pd.to_datetime(df_bandoc['Ngay_sinh'], errors='coerce')
df_bandoc['Ngay_cap'] = pd.to_datetime(df_bandoc['Ngay_cap'], errors='coerce')
df_bandoc['Ngay_het_han'] = pd.to_datetime(df_bandoc['Ngay_het_han'], errors='coerce')

df_bandoc['Ngay_sinh'] = df_bandoc['Ngay_sinh'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_bandoc['Ngay_cap'] = df_bandoc['Ngay_cap'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
df_bandoc['Ngay_het_han'] = df_bandoc['Ngay_het_han'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)

print(df_bandoc[['Ngay_sinh', 'Ngay_cap', 'Ngay_het_han']])

C:\Users\admin\AppData\Local\Temp\ipykernel_20680\3773408894.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_library)
C:\Users\admin\AppData\Local\Temp\ipykernel_20680\3773408894.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_bandoc['Ngay_sinh'] = pd.to_datetime(df_bandoc['Ngay_sinh'], errors='coerce')
C:\Users\admin\AppData\Local\Temp\ipykernel_20680\3773408894.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_bandoc['Ngay_cap'] = pd.to_datetime(df_bandoc['Ngay_cap'], errors='coerce')
C:\Users\adm

       Ngay_sinh  Ngay_cap  Ngay_het_han
0              0         0             0
1       19890502  20080926      20090831
2       19760101  20121227      20131227
3       19760101  20121227      20131227
4       19920215  20121228      20131228
...          ...       ...           ...
71079          0  20040106      20040831
71080          0  20020924      20030725
71081          0  20030826      20040831
71082   19800531  20040105      20040831
71083   19820806  20031129      20040831

[71084 rows x 3 columns]


## Xử lý Nien_khoa

In [72]:
df_bandoc['Khoa_hoc'] = df_bandoc['Khoa_hoc'].str.replace(r'\s*-\s*', '-', regex=True)
df_bandoc['Khoa_hoc'] = df_bandoc['Khoa_hoc'].replace('', np.nan)
#df_bandoc = df_bandoc.dropna(subset=['Khoa_hoc'])

query_Nienkhoa = "SELECT ID_nien_khoa, Ten_nien_khoa FROM olap.DIM_Nien_khoa"
df_nienkhoa = pd.read_sql(query_Nienkhoa, conn_dwh_library)

df_bandoc = df_bandoc.merge(df_nienkhoa,
            how='left',
            left_on='Khoa_hoc',
            right_on='Ten_nien_khoa')

# Nếu không khớp thì cho ID_nien_khoa = 0
df_bandoc['ID_nien_khoa'] = df_bandoc['ID_nien_khoa'].fillna(0).astype(int)

print(df_bandoc['ID_nien_khoa'])

0          0
1          0
2          0
3          0
4        131
        ... 
71079      1
71080      0
71081     35
71082      1
71083     44
Name: ID_nien_khoa, Length: 71084, dtype: int64


C:\Users\admin\AppData\Local\Temp\ipykernel_20680\633874014.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nienkhoa = pd.read_sql(query_Nienkhoa, conn_dwh_library)


## Xử lý Dan_toc

In [73]:
# đọc dữ liệu lấy từ bộ về
df_Data_Dim_Dan_toc = pd.read_csv("./data_dan_toc.csv")
df_Data_Dim_Dan_toc = df_Data_Dim_Dan_toc.where(pd.notnull(df_Data_Dim_Dan_toc), None)
# đọc dữ liệu đã lưu trong sql server
query_Dan_toc = "SELECT Id, dbo.DecodeUTF8String(Dan_toc) AS Dan_toc FROM Dan_toc "
df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)

#tìm và mapping 2 bảng lại
df_mapping = df_Dan_toc.copy()
df_mapping['Mapping Mã'] = None
df_mapping['CSV Mã'] = df_Data_Dim_Dan_toc['Mã']
df_mapping['CSV Tên'] = df_Data_Dim_Dan_toc['Tên']
df_mapping['CSV Tên khác'] = df_Data_Dim_Dan_toc['Tên gọi khác'].str.lower()
for i, dan_toc in enumerate(df_mapping['Dan_toc']):
    dan_toc = dan_toc.lower()
    dan_toc_bogach = dan_toc.replace("-", " ")
    dan_toc_botrong = dan_toc.replace(" ", "-")

    for j, row in df_mapping.iterrows():
        CSV_ten = row['CSV Tên'].lower() if pd.notna(row['CSV Tên']) else ""
        CSV_ten_khac = row['CSV Tên khác'].lower() if pd.notna(row['CSV Tên khác']) else ""
        if ((pd.notna(CSV_ten) and dan_toc == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc in CSV_ten_khac) or 
            (pd.notna(CSV_ten) and dan_toc_bogach == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_bogach in CSV_ten_khac) or
            (pd.notna(CSV_ten) and dan_toc_botrong == CSV_ten) or 
            (pd.notna(CSV_ten_khac) and dan_toc_botrong in CSV_ten_khac)):
            df_mapping.at[i, 'Mapping Mã'] = row['CSV Mã']
            break
        else:
            df_mapping.at[i, 'Mapping Mã'] = 56
print(df_mapping)

    Id  Dan_toc Mapping Mã  CSV Mã CSV Tên  \
0    1     Kinh        1.0     1.0    Kinh   
1    2    Mường        3.0     2.0     Tày   
2    3      Tày        2.0     3.0    Thái   
3    4     Thái        3.0     4.0     Hoa   
4    5      Hoa        4.0     5.0  Khơ-me   
..  ..      ...        ...     ...     ...   
68  71     Ê Đê       12.0     NaN     NaN   
69  72      Thổ        2.0     NaN     NaN   
70  73    Kờ Ho         56     NaN     NaN   
71  74     Jrai         56     NaN     NaN   
72  75  Châu mạ       28.0     NaN     NaN   

                                         CSV Tên khác  
0                                                việt  
1           thổ, ngạn, phén, thù lao, pa dí, tày khao  
2   tày đăm, tày mười, tày thanh, mán thanh, hàng ...  
3   hán, triều châu, phúc kiến, quảng đông, hải na...  
4              cur, cul, cu, thổ, việt gốc miên, krôm  
..                                                ...  
68                                                NaN  

C:\Users\admin\AppData\Local\Temp\ipykernel_20680\2376671368.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Dan_toc = pd.read_sql(query_Dan_toc, conn_libol)


In [74]:
map_dict = dict(zip(df_mapping['Id'], df_mapping['Mapping Mã'])) # Tạo map_dict để ánh xạ từ Id sang Mapping_Ma trong df_mapping

for index, row in df_bandoc.iterrows(): # Lặp qua từng dòng trong df_ban_doc để cập nhật Dan_toc_ID
    
    if not pd.isna(row['Dan_toc_ID']): # Kiểm tra nếu Dan_toc_ID rỗng (None hoặc NaN)
        if row['Dan_toc_ID'] in map_dict: # Kiểm tra nếu Dan_toc_ID có trong map_dict
            df_bandoc.at[index, 'Dan_toc_ID'] = map_dict[row['Dan_toc_ID']]# Nếu tìm thấy, thay thế bằng giá trị Mapping_Ma
        else:
            df_bandoc.at[index, 'Dan_toc_ID'] = 56 # Nếu không tìm thấy, gán Dan_toc_ID bằng 0
    else:
        df_bandoc.at[index, 'Dan_toc_ID'] = 56
print(df_bandoc['Dan_toc_ID'])

0        56
1         1
2         1
3         1
4         1
         ..
71079     1
71080     1
71081     1
71082     1
71083     1
Name: Dan_toc_ID, Length: 71084, dtype: int64


## Xử lý Trinh_do

In [75]:
query_Trinhdo = "SELECT ID_trinh_do FROM olap.DIM_Trinh_do"
df_trinhdo = pd.read_sql(query_Trinhdo, conn_dwh_library)

trinhdo_ids = set(df_trinhdo['ID_trinh_do'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong trinhdo_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Trinh_do_ID'] = df_bandoc['Trinh_do_ID'].apply(lambda x: x if x in trinhdo_ids else 0)

print(df_bandoc['Trinh_do_ID'])

0        0
1        4
2        9
3        9
4        9
        ..
71079    9
71080    9
71081    9
71082    4
71083    9
Name: Trinh_do_ID, Length: 71084, dtype: int64


C:\Users\admin\AppData\Local\Temp\ipykernel_20680\1481478981.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_trinhdo = pd.read_sql(query_Trinhdo, conn_dwh_library)


## Xử lý Lop

In [76]:
query_lop = "SELECT ID_lop FROM olap.DIM_Lop"
df_lop = pd.read_sql(query_lop, conn_dwh_library)

lop_ids = set(df_lop['ID_lop'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong lop_ids hay không. 
# Nếu có, chuyển sang dạng chữ in hoa (upper()), nếu không, gán giá trị bằng 0.
df_bandoc['Lop'] = df_bandoc['Lop'].str.upper()
df_bandoc['Lop'] = df_bandoc['Lop'].apply(lambda x: x.upper() if x in lop_ids else 0)

print(df_bandoc['Lop'])

0              0
1        081070B
2        1200016
3        1200016
4              0
          ...   
71079    0040IVL
71080          0
71081    021011B
71082    00401VL
71083          0
Name: Lop, Length: 71084, dtype: object


C:\Users\admin\AppData\Local\Temp\ipykernel_20680\529711181.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lop = pd.read_sql(query_lop, conn_dwh_library)


## Xử lý Nhom_ban_doc

In [77]:
query_Nhombandoc = "SELECT ID_nhom_ban_doc FROM olap.DIM_Nhom_ban_doc"
df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_dwh_library)

nhombandoc_ids = set(df_nhombandoc['ID_nhom_ban_doc'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong nhombandoc_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Nhom_ID'] = df_bandoc['Nhom_ID'].apply(lambda x: x if x in nhombandoc_ids else 0)

print(df_bandoc['Nhom_ID'])

0         0
1        14
2        12
3        24
4        12
         ..
71079    12
71080     9
71081     9
71082    12
71083     9
Name: Nhom_ID, Length: 71084, dtype: int64


C:\Users\admin\AppData\Local\Temp\ipykernel_20680\3965989656.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhombandoc = pd.read_sql(query_Nhombandoc, conn_dwh_library)


## Xử lý Nhom_nghanh_nghe

In [78]:
query_Nhomnghanhnghe = "SELECT ID_nhom_nghanh_nghe FROM olap.DIM_Nhom_nghanh_nghe"
df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_dwh_library)

nhomnghanhnghe_ids = set(df_nhomnghanhnge['ID_nhom_nghanh_nghe'])

# Duyệt qua từng giá trị Lop trong df_bandoc và kiểm tra xem nó có nằm trong nhomnghanhnghe_ids hay không. 
# Nếu có thì dữ nguyên nếu không, gán giá trị bằng 0.
df_bandoc['Nhom_nghanh_nghe_ID'] = df_bandoc['Nhom_nghanh_nghe_ID'].apply(lambda x: x if x in nhomnghanhnghe_ids else 0)

print(df_bandoc['Nhom_nghanh_nghe_ID'])

0          0.0
1        100.0
2         59.0
3         59.0
4         59.0
         ...  
71079    125.0
71080     50.0
71081     65.0
71082    125.0
71083    106.0
Name: Nhom_nghanh_nghe_ID, Length: 71084, dtype: float64


C:\Users\admin\AppData\Local\Temp\ipykernel_20680\938135672.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_nhomnghanhnge = pd.read_sql(query_Nhomnghanhnghe, conn_dwh_library)


## Xử lý duplicate cho primary key

In [79]:
df_bandoc['So_the'] = df_bandoc['So_the'].astype(str).str.strip()
df_bandoc = df_bandoc.drop_duplicates(subset='So_the').reset_index(drop=True) 
print(df_bandoc)

           So_the              Ho_ten  Ngay_sinh  Dan_toc_ID  Trinh_do_ID  \
0               0    (Không xác định)          0          56            0   
1       0/8107029         Vũ Minh Đức   19890502           1            4   
2        00000001      LƯU VĨNH QUANG   19760101           1            9   
3        00000002       PHẠM HỮU TIẾN   19920215           1            9   
4        00000003  NGUYỄN THỦY THƯƠNG   19701206           1            9   
...           ...                 ...        ...         ...          ...   
71055   ÊU0401079         LÊ VĂN HIẾU          0           1            9   
71056    Ô0105040          NG. TẤN ĐỘ          0           1            9   
71057   ên2101158        ĐÀO MINH ĐỨC          0           1            9   
71058   ƠI4401092        PHẠM VĂN LỢI   19800531           1            4   
71059  ƠNG1113054    LÊ THỊ MỸ PHƯỢNG   19820806           1            9   

      So_dien_thoai Nghe_nghiep        Co_quan chuc_vu  \
0              No

## Load data

### [Nếu cần] Clear bảng

In [80]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Ban_doc"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [81]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_library.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO olap.DIM_Ban_doc (
                    ID_ban_doc, Ho_ten, Ngay_sinh, 
                    ID_nien_khoa, ID_dan_toc, ID_trinh_do,
                    So_dien_thoai, Nghe_nghiep, Co_quan, Chuc_vu,
                    Dia_chi_tam_tru, Dia_chi_thuong_tru, ID_lop,
                    Anh, Ngay_cap, Ngay_het_han, Email, ID_nhom_ban_doc,
                    ID_nhom_nghanh_nghe, Gioi_tinh, Tinh_trang, Ghi_chu, Mat_khau
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """

# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['So_the'], row['Ho_ten'], row['Ngay_sinh'], 
        row['ID_nien_khoa'], row['Dan_toc_ID'], row['Trinh_do_ID'],
        row['So_dien_thoai'], row['Nghe_nghiep'], row['Co_quan'], row['chuc_vu'],
        row['Dia_chi'], row['Dia_chi_thuong_tru'], row['Lop'],
        row['Anh'], row['Ngay_cap'], row['Ngay_het_han'], row['Email'], row['Nhom_ID'],
        row['Nhom_nghanh_nghe_ID'], row['Gioi_tinh'], row['Status'], row['Ghi_chu'], row['Mat_khau']
    )
    for index, row in df_bandoc.iterrows()
]

cursor_dwh.executemany(insert_query, data_to_insert) # Sử dụng executemany để chèn dữ liệu cùng lúc
conn_dwh_library.commit() # Commit thay đổi
cursor_dwh.close() # Đóng cursor và kết nối
conn_dwh_library.close()